In [5]:
import torch
import torch.nn as nn
import torch.optim as optim

USER_NO = 9  # Number of users
BS_NO = 2    # Number of base stations

# Define synthetic channel gain where all users are close to the first BS

# channel_gain = torch.zeros(USER_NO, BS_NO)
# channel_gain[:, 0] = torch.rand(USER_NO) * 0.5 + 0.5  # Higher values for the first BS (close range)
# for i in range(1, BS_NO):
#     channel_gain[:, i] = torch.rand(USER_NO) * 0.5  # Lower values for other BSs (farther range)

# channel_gain = torch.rand(USER_NO, BS_NO)  # Random values between 0 and 1

# Define the input sizes
channel_gain = torch.tensor([
    [1.01243975e-08, 1.89566844e-08, 1.18546492e-08, 1.64374138e-08, 4.40855244e-08, 9.93131941e-09, 3.09362772e-09, 1.30197957e-08, 4.88414007e-09],
    [3.81201400e-09, 2.68718961e-08, 4.13384146e-09, 6.14672592e-09, 5.72870100e-08, 5.42210154e-09, 1.01222581e-07, 2.89773752e-09, 1.19526998e-08]
], dtype=torch.float32)

# Transpose the channel gain matrix to match the expected dimensions (USER_NO * BS_NO)
channel_gain = channel_gain.T

###############################################################

# Create a previous assignment matrix where each user is connected to exactly one BS
# prev_assignment = torch.zeros(USER_NO, BS_NO)
# for user in range(USER_NO):
#     bs = torch.randint(0, BS_NO, (1,))
#     prev_assignment[user, bs] = 1.0

prev_assignment = torch.tensor([
    [1., 0.],
    [0., 1.],
    [1., 0.],
    [1., 0.],
    [0., 1.],
    [1., 0.],
    [0., 1.],
    [1., 0.],
    [0., 1.]
], dtype=torch.float32)

# Create a matrix of user satisfaction from the previous timestep
# prev_satisfaction = torch.rand(USER_NO)  # Random values between 0 and 1
# User satisfaction from the last timestep
prev_satisfaction = torch.tensor([0.29790624, 0.97148202, 0.51188557, 0.884983, 0.99935268, 0.99999383, 1.0, 0.99999936, 1.0], dtype=torch.float32)


# Flatten the inputs and concatenate
input_data = torch.cat([channel_gain.flatten(), prev_satisfaction, prev_assignment.flatten()]).unsqueeze(0)
input_size = input_data.size(1)  # Adjust to match the feature size (batch size is now the first dimension)

class UserAssignmentMLP(nn.Module):
    def __init__(self, input_size, hidden_sizes, output_size):
        super(UserAssignmentMLP, self).__init__()
        self.hidden_layers = nn.ModuleList()
        for i, hidden_size in enumerate(hidden_sizes):
            self.hidden_layers.append(nn.Linear(input_size if i == 0 else hidden_sizes[i-1], hidden_size))
            self.hidden_layers.append(nn.ReLU())
        self.output_layer = nn.Linear(hidden_sizes[-1], output_size)
        self.output_activation = nn.Softmax(dim=1)  # Use softmax activation for probabilistic assignment

    def forward(self, x):
        for i, layer in enumerate(self.hidden_layers):
            x = layer(x)
        x = self.output_layer(x)
        x = self.output_activation(x)
        return x
    
    def custom_loss(self, output, target, channel_gain, prev_assignment, prev_satisfaction):
        bce_loss = nn.BCELoss()(output, target)
        gain_loss = -torch.mean(channel_gain * output)

        # Penalize changes to assignments if user was satisfied in previous timestep
        assignment_change = torch.abs(output - prev_assignment)
        satisfaction_penalty = torch.mean(assignment_change * prev_satisfaction.unsqueeze(1))
        
        # Fairness penalty to distribute users across BSs
        fairness_penalty = torch.mean(torch.sum(output, dim=0) ** 2)  # Squared sum to penalize high concentrations
        
        # Penalty for not assigning each user to any BS
        assignment_penalty = torch.mean((torch.sum(output, dim=1) - 1) ** 2)
        
        return bce_loss + 0.1 * gain_loss + 0.01 * satisfaction_penalty + 0.1 * fairness_penalty + 0.1 * assignment_penalty
    
    def train_model(self, input_data, channel_gain, prev_assignment, prev_satisfaction, num_epochs=100, learning_rate=0.001):
        optimizer = optim.Adam(self.parameters(), lr=learning_rate)
        for epoch in range(num_epochs):
            self.train()
            optimizer.zero_grad()
            
            # Forward pass
            output = self(input_data)
            output = output.view(USER_NO, BS_NO)
            
            # Normalize channel gain and satisfaction to balance their effects
            norm_channel_gain = (channel_gain - channel_gain.min()) / (channel_gain.max() - channel_gain.min())
            norm_satisfaction = (prev_satisfaction - prev_satisfaction.min()) / (prev_satisfaction.max() - prev_satisfaction.min())
            
            # Combine metrics with equal weights
            combined_metric = norm_channel_gain + norm_satisfaction.unsqueeze(1)
            target = (combined_metric == combined_metric.max(1, keepdim=True)[0]).float()
            
            # Calculate custom loss
            loss = self.custom_loss(output, target, channel_gain, prev_assignment, prev_satisfaction)
            
            # Backward pass and optimization
            loss.backward()
            optimizer.step()
            
            if (epoch + 1) % 10 == 0:
                print(f'Epoch [{epoch + 1}/{num_epochs}], Loss: {loss.item():.4f}')
    
    def post_process(self, input_data, channel_gain):
        with torch.no_grad():
            self.eval()
            output = self(input_data).view(USER_NO, BS_NO)
            # Binarize the output
            assignment = (output > 0.5).float()
            
            # Ensure each user connects to only one BS
            for user in range(USER_NO):
                if assignment[user].sum() != 1:
                    _, best_bs = channel_gain[user].max(0)
                    assignment[user] = torch.zeros(BS_NO)
                    assignment[user, best_bs] = 1
        return assignment

# Create an instance of the model
hidden_sizes = [128, 64]  # Example hidden layer sizes
output_size = USER_NO * BS_NO  # Output is the flattened user assignment matrix
model = UserAssignmentMLP(input_size, hidden_sizes, output_size)

# Train the model
model.train_model(input_data, channel_gain, prev_assignment, prev_satisfaction, learning_rate=0.001)

# Post-process the results
final_assignment = model.post_process(input_data, channel_gain)

print("Channel Gain Matrix:\n", channel_gain)
print("Previous Assignment Matrix:\n", prev_assignment)
print("Previous Satisfaction Levels:\n", prev_satisfaction)
print("Input Data:\n", input_data)
print("Final Assignment:\n", final_assignment)


Epoch [10/200], Loss: 1.4999
Epoch [20/200], Loss: 1.3736
Epoch [30/200], Loss: 1.2626
Epoch [40/200], Loss: 1.2207
Epoch [50/200], Loss: 1.2112
Epoch [60/200], Loss: 1.2083
Epoch [70/200], Loss: 1.2075
Epoch [80/200], Loss: 1.2073
Epoch [90/200], Loss: 1.2072
Epoch [100/200], Loss: 1.2071
Epoch [110/200], Loss: 1.2070
Epoch [120/200], Loss: 1.2070
Epoch [130/200], Loss: 1.2070
Epoch [140/200], Loss: 1.2069
Epoch [150/200], Loss: 1.2069
Epoch [160/200], Loss: 1.2069
Epoch [170/200], Loss: 1.2069
Epoch [180/200], Loss: 1.2069
Epoch [190/200], Loss: 1.2068
Epoch [200/200], Loss: 1.2068
Channel Gain Matrix:
 tensor([[1.0124e-08, 3.8120e-09],
        [1.8957e-08, 2.6872e-08],
        [1.1855e-08, 4.1338e-09],
        [1.6437e-08, 6.1467e-09],
        [4.4086e-08, 5.7287e-08],
        [9.9313e-09, 5.4221e-09],
        [3.0936e-09, 1.0122e-07],
        [1.3020e-08, 2.8977e-09],
        [4.8841e-09, 1.1953e-08]])
Previous Assignment Matrix:
 tensor([[1., 0.],
        [0., 1.],
        [1., 0.